# 04 · Đánh giá mô hình và phân tích lỗi (Model Evaluation & Error Analysis)

Mục tiêu của notebook:
1. **Đánh giá toàn diện trên tập dữ liệu kiểm thử (Test Split)**:
   - Chỉ số đếm (Counting Metrics): MAE, RMSE, MAPE (%), Bias, $R^2$, Sai số tổng.
   - Chỉ số phát hiện (Detection Metrics): mAP@0.5, mAP@0.75, mAP@[0.5:0.95], Precision, Recall, F1.
   - Hiệu năng tốc độ: FPS, Latency (ms), Số lượng tham số (M params).
2. **Tối ưu hóa ngưỡng đếm (Count Threshold Tuning)**:
   - Lựa chọn ngưỡng tin cậy (score threshold) tối ưu trên tập Validation rồi áp dụng sang Test để tránh rò rỉ dữ liệu (data leakage).
3. **Thử nghiệm Kỹ thuật Cắt ô Tiling (SAHI-style)**:
   - Đánh giá xem kỹ thuật chia nhỏ ảnh (slicing) có cải thiện số đếm cho quả nhỏ / ảnh độ phân giải cao hay không.
4. **Phân tích lỗi chuyên sâu (Error Analysis)**:
   - Sai số theo từng phân khúc mật độ quả (`error_by_count_bin`): Trả lời câu hỏi *Detection vs Density khi quả nhiều/dày đặc*.
   - Tìm và trực quan hóa các trường hợp sai lệch nhiều nhất (Worst-case Analysis).

In [ ]:
import os, sys, json, time, shutil, warnings
from pathlib import Path

# ---- Xác định thư mục gốc project: chạy được cả local lẫn Kaggle ----
IN_KAGGLE = Path('/kaggle/working').exists()
if IN_KAGGLE:
    CODE_INPUT = Path('/kaggle/input/fruitcount-code/FruitCount')   # sửa theo tên Kaggle Dataset chứa code
    ROOT = Path('/kaggle/working/FruitCount')
    if not ROOT.exists():
        shutil.copytree(CODE_INPUT, ROOT)
    OUT_BASE = Path('/kaggle/working')
else:
    ROOT = Path.cwd().resolve()
    while ROOT != ROOT.parent and not (ROOT / 'src').is_dir():
        ROOT = ROOT.parent
    OUT_BASE = ROOT / 'outputs'
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image as IPImage

from src.utils import load_yaml, save_yaml, read_json, write_json, load_image, ensure_dir

DS_CFG_PATH = Path(os.environ.get('FRUITCOUNT_DATASET_CFG', ROOT / 'configs/dataset.yaml'))
DS_CFG = load_yaml(DS_CFG_PATH)
PROCESSED = ROOT / DS_CFG.get('processed_dir', 'data/processed')
RUNS = Path(os.environ.get('FRUITCOUNT_RUNS', OUT_BASE / 'runs' if IN_KAGGLE else ROOT / 'runs'))
OUT_DIR = ensure_dir(OUT_BASE / 'notebooks' / '04_evaluate')   # hình + bảng của notebook này

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})
FAMILY_COLORS = {'detection': '#4C72B0', 'segmentation': '#55A868', 'density': '#C44E52'}


def savefig(fig, name):
    fig.savefig(OUT_DIR / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()


class StopExecution(Exception):
    """Dừng 'Run All' một cách êm (không in traceback)."""
    def _render_traceback_(self):
        return []


try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

print(f'Môi trường : {"Kaggle" if IN_KAGGLE else "Local"}')
print(f'ROOT       : {ROOT}')
print(f'PROCESSED  : {PROCESSED}')
print(f'RUNS       : {RUNS}')
print(f'OUT_DIR    : {OUT_DIR}')

## 1. Tham số Đánh giá

In [ ]:
EVAL_SPLIT = 'test'         # 'test' hoặc 'val'
MEASURE_SPEED = True        # Đo tốc độ FPS / latency
RUN_TILING_EXPERIMENT = True # Chạy thêm phiên bản có tiling cho mô hình phát hiện (YOLO)
BATCH_SIZE = 4

# Danh sách các mô hình cần đánh giá (tìm trong thư mục RUNS)
available_models = [d.name for d in RUNS.iterdir() if d.is_dir() and (d / 'weights' / 'best.pt').exists()]
print("Các mô hình đã có trọng số (weights/best.pt):", available_models)

if not available_models:
    print("⚠️ Chưa tìm thấy checkpoint nào trong", RUNS)
    print("Vui lòng huấn luyện ít nhất 1 mô hình ở 03_train.ipynb trước, hoặc chỉ định đường dẫn RUNS chính xác.")

In [ ]:
import subprocess


def run(cmd, log_file=None):
    """Chạy lệnh con, in log trực tiếp (và ghi ra file nếu có). Trả về mã thoát."""
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd), flush=True)
    f = open(log_file, 'a', encoding='utf-8') if log_file else None
    try:
        p = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
        for line in p.stdout:
            print(line, end='')
            if f:
                f.write(line)
        p.wait()
    finally:
        if f:
            f.close()
    if p.returncode != 0:
        print(f'⚠️ Lệnh kết thúc với mã {p.returncode}')
    return p.returncode

## 2. Thực thi Đánh giá (Evaluate Runner)
Gọi `scripts/evaluate.py` cho từng mô hình có sẵn.
Kết quả sẽ được ghi vào: `runs/<model>/metrics_test.json`, `per_image_test.csv`, `error_by_count_test.csv`.

In [ ]:
eval_outputs = {}

for m in available_models:
    cfg_file = RUNS / m / 'config.yaml'
    if not cfg_file.exists():
        continue
    
    print(f"\n>>> Đang đánh giá mô hình: {m} (Split: {EVAL_SPLIT})")
    cmd = [
        'python', 'scripts/evaluate.py',
        '--model', str(cfg_file),
        '--out', str(RUNS),
        '--split', EVAL_SPLIT,
        '--batch-size', str(BATCH_SIZE)
    ]
    if MEASURE_SPEED:
        cmd.append('--speed')
        
    ret = run(cmd)
    
    # Nếu là mô hình detection (YOLO), chạy thêm thử nghiệm với Tiling
    if RUN_TILING_EXPERIMENT and 'yolo' in m:
        print(f">>> Đang đánh giá {m} với chế độ TILING (SAHI-style)...")
        cmd_tile = [
            'python', 'scripts/evaluate.py',
            '--model', str(cfg_file),
            '--out', str(RUNS),
            '--split', EVAL_SPLIT,
            '--tile',
            '--tag', '_tile'
        ]
        run(cmd_tile)

print("\nHoàn tất quá trình đánh giá!")

## 3. Tổng hợp Bảng Kết quả Đánh giá
Đọc tất cả các file `metrics_<split>*.json` và hiển thị dưới dạng bảng so sánh chuẩn hóa.

In [ ]:
from src.utils.run_layout import evaluation_reports, report_model_dir
# Reports are loaded directly below; no external common module required.

reports, per_image = {}, {}
for f in sorted(evaluation_reports(RUNS, EVAL_SPLIT)):
    r = read_json(f)
    tag = f.stem[len(f'metrics_{EVAL_SPLIT}'):]
    label = r['model'] + (' (tile)' if r.get('tiling') else '')
    r['_dir'], r['_tag'] = report_model_dir(f), tag
    reports[label] = r
    pi = f.parent / f'per_image_{EVAL_SPLIT}{tag}.csv'
    if pi.exists():
        per_image[label] = pd.read_csv(pi, dtype={'image_id': str})

summary_rows = []
for name, r in reports.items():
    c = r.get('counting', {})
    d = r.get('detection', {})
    s = r.get('speed', {})
    summary_rows.append({
        'Model': name,
        'Family': r.get('family', ''),
        'MAE': c.get('mae', np.nan),
        'RMSE': c.get('rmse', np.nan),
        'MAPE (%)': c.get('mape_pct', np.nan),
        'Bias': c.get('bias', np.nan),
        'R2': c.get('r2', np.nan),
        'mAP@0.5': d.get('map50', np.nan),
        'mAP@[0.5:0.95]': d.get('map', np.nan),
        'FPS': s.get('fps', np.nan),
        'Params (M)': s.get('params_million', np.nan),
        'Best Threshold': r.get('threshold_tuning', {}).get('best_threshold', r.get('count_threshold', np.nan))
    })

df_eval_summary = pd.DataFrame(summary_rows).sort_values('MAE').reset_index(drop=True)
display(df_eval_summary.round(3))

## 4. Biểu đồ Dự đoán vs Thực tế (Pred vs GT Scatter Plot)
Đường chéo $y = x$ thể hiện dự đoán hoàn hảo. Các điểm phân tán quanh đường này cho biết độ lệch và phương sai của từng mô hình.

In [ ]:
n_models = len(reports)
if n_models > 0:
    fig, axes = plt.subplots(1, n_models, figsize=(5.5 * n_models, 5), squeeze=False)
    for idx, (name, r) in enumerate(reports.items()):
        ax = axes[0][idx]
        df_p = per_image.get(name)
        if df_p is not None:
            ax.scatter(df_p['gt'], df_p['pred'], alpha=0.6, s=20, color='#4C72B0')
            max_val = max(df_p['gt'].max(), df_p['pred'].max()) * 1.05
            ax.plot([0, max_val], [0, max_val], 'r--', label='y = x')
            ax.set(
                title=f"{name}\nMAE={r['counting']['mae']:.2f} | R2={r['counting']['r2']:.3f}",
                xlabel="Số quả thực tế (GT)",
                ylabel="Số quả dự đoán (Pred)",
                xlim=[0, max_val],
                ylim=[0, max_val]
            )
            ax.legend()
    plt.tight_layout()
    savefig(fig, "pred_vs_gt_comparison")

## 5. Phân tích Sai số theo Mật độ quả (Error by Density Bins)
Câu hỏi trọng tâm: *Khi số lượng quả trên cây tăng lên (mật độ dày đặc, chồng lấn cao), sai số MAE của họ Detection và họ Density thay đổi như thế nào?*

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for name, r in reports.items():
    bin_data = r.get('error_by_count_bin', [])
    if bin_data:
        df_b = pd.DataFrame(bin_data)
        # Sắp xếp các bins theo thứ tự tăng dần
        df_b['sort_key'] = df_b['gt_count_bin'].apply(lambda x: int(str(x).replace('>=', '').split('-')[0]))
        df_b = df_b.sort_values('sort_key')
        
        ax.plot(df_b['gt_count_bin'], df_b['mae'], marker='o', lw=2, label=f"{name} ({r['family']})")

ax.set(
    title=f"Sai số MAE theo nhóm số lượng quả ({EVAL_SPLIT.upper()} Set)",
    xlabel="Khoảng số quả trên ảnh (Ground Truth Bin)",
    ylabel="Sai số trung bình tuyệt đối (MAE)"
)
ax.legend()
plt.tight_layout()
savefig(fig, "mae_by_fruit_count_bins")

## 6. Phân tích các Trường hợp Sai lệch Nhiều nhất (Worst-Case Analysis)
Lấy ra 5 ảnh có sai số tuyệt đối lớn nhất $|Pred - GT|$ của mô hình có MAE tốt nhất để trực quan hóa và tìm nguyên nhân:
- Do quả bị che khuất quá nặng?
- Do quả quá nhỏ / quá xa?
- Do nhãn thiếu hoặc lỗi ánh sáng chói / tối?

In [ ]:
from src.data import load_split_records
from src.inference import CountingPredictor
from src.visualization import render_result, draw_count_banner

best_model_name = df_eval_summary.iloc[0]['Model']
base_cfg_name = reports[best_model_name]['model']
print(f"Khảo sát Worst Cases trên mô hình tốt nhất: {best_model_name}")

df_best_pi = per_image[best_model_name].sort_values('abs_error', ascending=False)
display(df_best_pi.head(10))

# Khởi tạo predictor
pred_obj = CountingPredictor.from_config(
    reports[best_model_name]['_dir'] / 'config.yaml',
    weights=reports[best_model_name]['weights']
)
if reports[best_model_name].get('count_threshold') is not None:
    pred_obj.set_count_threshold(reports[best_model_name]['count_threshold'])
if reports[best_model_name].get('tiling'):
    pred_obj.set_tiling(True)

# Lấy 4 ảnh sai lệch nhiều nhất
top_worst = df_best_pi.head(4)
test_recs = {r.image_id: r for r in load_split_records(PROCESSED, EVAL_SPLIT)}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (_, row) in enumerate(top_worst.iterrows()):
    rec = test_recs[row['image_id']]
    img = load_image(rec.image_path)
    res = pred_obj.predict_array(img)
    
    vis = render_result(img, res)
    vis = draw_count_banner(vis, f"GT: {int(row['gt'])} | Pred: {int(round(row['pred']))} | Diff: {int(row['error']):+d}")
    
    axes[idx].imshow(vis)
    axes[idx].set_title(f"{row['image_id']} (Abs Error = {row['abs_error']:.1f})", fontsize=11)
    axes[idx].axis('off')

plt.tight_layout()
savefig(fig, f"worst_cases_{base_cfg_name}")

## 7. Tổng kết & Nhận xét Đánh giá
- Bảng tổng hợp đã chỉ ra mô hình có MAE thấp nhất và tốc độ FPS cao nhất.
- Phân tích theo nhóm số quả đã minh họa rõ tính ưu/nhược của từng kiến trúc khi mật độ thay đổi.
- Kỹ thuật Tiling chứng minh sự tác động đối với các quả có kích thước nhỏ.
- Tiếp theo: Sang `05_compare_models.ipynb` để so sánh toàn diện giữa tất cả các mô hình, vẽ biểu đồ radar và trích xuất bảng Markdown phục vụ báo cáo.